In [1]:
import pandas as pd
import itertools
from collections import Counter
import numpy as np

df = pd.read_csv('/root/data/mimic-iv_data/hosp/diagnoses_icd.csv')


# Add ICD version to code
df['icd_code'] = df['icd_code'] + '-' + df['icd_version'].astype(str)

In [2]:
df

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000032,22595853,1,5723-9,9
1,10000032,22595853,2,78959-9,9
2,10000032,22595853,3,5715-9,9
3,10000032,22595853,4,07070-9,9
4,10000032,22595853,5,496-9,9
...,...,...,...,...,...
5006879,19999987,23865745,7,41401-9,9
5006880,19999987,23865745,8,78039-9,9
5006881,19999987,23865745,9,0413-9,9
5006882,19999987,23865745,10,36846-9,9


In [3]:
unique_codes = sorted(set(df['icd_code']))
len(unique_codes)

26164

In [4]:
# Checking special token IDs for BERT

# from transformers import BertTokenizer
# tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# text = "[CLS][PAD] [UNK] [SEP] This is a test sentence. and"
# encoding = tokenizer.encode_plus(text, add_special_tokens = True,    truncation = True, return_tensors = "pt")
# print(encoding)

In [5]:
# Use BERT tokenizer settings

# Get special tokens
og_bert_token_path = "/root/tia/bert-on-mimic-iv/data/og_bert_tokens.txt"
special_tokens = []
with open(og_bert_token_path, 'r') as f:
     lines = f.readlines()
     for line in lines:
        special_tokens.append(line.strip())

vocab = {str(code): i for i, code in enumerate(unique_codes, start=len(special_tokens))}

# Add special tokens to vocab
for i in range(len(special_tokens)):
    vocab[special_tokens[i]] = i


import json
with open('./data/vocab.json', 'w') as f:
    json.dump(vocab, f)

In [6]:
with open('./data/icd_9_10_vocab.txt', 'w') as f:
    f.write('\n'.join(unique_codes))

In [7]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace, WhitespaceSplit

vocab = WordLevel.read_file("./data/vocab.json")
tokenizer = Tokenizer(WordLevel(vocab, unk_token="[UNK]"))
tokenizer.pre_tokenizer = WhitespaceSplit()

In [8]:
## Note: use WhitespaceSplit instead of Whistespace
ws = WhitespaceSplit()
w = Whitespace()
ws.pre_tokenize_str('Hello-world'), w.pre_tokenize_str("Hello-world")

([('Hello-world', (0, 11))],
 [('Hello', (0, 5)), ('-', (5, 6)), ('world', (6, 11))])

In [9]:
tokenizer.token_to_id('0020-9'), tokenizer.token_to_id('Z9989-10'), tokenizer.token_to_id('[unused0]')

(104, 26267, 1)

In [10]:
tokenizer.token_to_id('[PAD]'), tokenizer.token_to_id('[SEP]'), tokenizer.token_to_id('[CLS]'),  tokenizer.token_to_id('[UNK]'), tokenizer.token_to_id('5')

(0, 102, 101, 100, None)

In [11]:
from tokenizers.processors import BertProcessing
tokenizer.post_processor = BertProcessing(
    sep=('[SEP]', tokenizer.token_to_id('[SEP]')), 
    cls=('[CLS]', tokenizer.token_to_id('[CLS]')))

In [12]:
tokenizer.save('./data/tokenizer-icd.json')

In [20]:
from transformers import PreTrainedTokenizerFast

t = PreTrainedTokenizerFast(tokenizer_file='./data/tokenizer-icd.json')
t.add_special_tokens({
    'pad_token': '[PAD]',
    'mask_token': '[MASK]',
    'unk_token': '[UNK]',
    'cls_token': '[CLS]',
    'sep_token': '[SEP]',
    'additional_special_tokens': [f'[unused{i}]' for i in range(99)]
})

0

In [21]:
t.save_pretrained('./data/tokenizer-icd-final')

('./data/tokenizer-icd-final/tokenizer_config.json',
 './data/tokenizer-icd-final/special_tokens_map.json',
 './data/tokenizer-icd-final/tokenizer.json')

In [16]:
text = unique_codes[:200]
encoding = t(
            text,
            return_offsets_mapping=False,
            padding='max_length',
            truncation=True,
            max_length=128,
            return_overflowing_tokens=False)

In [22]:
t(unique_codes[1000:1012], is_split_into_words=True, padding='max_length', max_length=128)

{'input_ids': [101, 1104, 1105, 1106, 1107, 1108, 1109, 1110, 1111, 1112, 1113, 1114, 1115, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [18]:
t(unique_codes[1])

{'input_ids': [101, 105, 102], 'token_type_ids': [0, 0, 0], 'attention_mask': [1, 1, 1]}